<img src='https://docs.abstractsecurity.app/img/logos/logo-dark.svg' height='40'>

# Abstract AI-SOC Notebook — threat hunting, blast radius, prediction & write-back

A branded, interactive analyst workspace over the Abstract model. Entity graphs, MITRE
heat maps, attack timelines, blast radius, continuous risk, identity/NHI/agent taxonomy,
OSINT enrichment, what-if/replay — then **generate a report** and **write back to Abstract**.

> Reuses the CLI-verified engine (`pipeline.py`), live REST adapter (`abstract_client.py`),
> visualizations (`viz.py`), and identity/OSINT registry (`identities.py`).

## 0. Setup

In [ ]:
# one-time: install interactive viz libs (engine + networkx already present)
# %pip install matplotlib networkx numpy
%matplotlib inline

In [ ]:
from pipeline import (normalize, Graph, run_detections, run_investigation,
                      continuous_scores, efficiency)
from data import events, IOCS, INCIDENT_START
import viz, identities as ID

raw = events(); norm = [normalize(i, r) for i, r in enumerate(raw)]
g = Graph()
for ev in norm: g.add(ev)
findings = run_detections(norm, IOCS)
inv = run_investigation(g, findings, IOCS, INCIDENT_START, norm)
scores = continuous_scores(norm, IOCS)
metrics = efficiency(norm, findings)
print(f'{len(norm):,} events | {len(g.nodes)} entities | {len(findings)} findings')

## 1. Connect to the live tenant (read) — optional
Needs a key in `~/.abstract.env`. Falls back to offline if absent.

In [ ]:
client = mitre_tactics = None
try:
    from abstract_client import AbstractClient
    client = AbstractClient('api'); print('connect:', client.connect())
    mitre_tactics = (client._req('GET','/v3/rules/mitre').get('body') or {}).get('tactics')
    print('live MITRE tactics:', len(mitre_tactics or []))
except Exception as e:
    print('offline (no key):', e)

## 2. Campaign entity graph
User / host / account / NHI / AI-agent / IOC — sized by continuous risk.

In [ ]:
viz.draw_entity_graph(g, IOCS, scores);

## 3. Continuous entity risk + prediction

In [ ]:
viz.draw_risk_bars(scores)
print('PREDICTED next targets:', inv['prediction']['predicted_next_targets'])

## 4. MITRE ATT&CK coverage heat map (live if connected)

In [ ]:
tactics = mitre_tactics or [
  {'name':'Initial Access','total':10,'enabled':9},{'name':'Execution','total':14,'enabled':13},
  {'name':'Persistence','total':19,'enabled':17},{'name':'Defense Evasion','total':42,'enabled':38},
  {'name':'Cred Access','total':17,'enabled':16},{'name':'C2','total':17,'enabled':16},
  {'name':'Exfiltration','total':9,'enabled':9}]
viz.draw_mitre_heatmap(tactics);

## 5. Attack-chain timeline

In [ ]:
viz.draw_attack_timeline(norm);

## 6. Blast radius — real-time vs LakeVilla replay, across all identity kinds

In [ ]:
sc = inv['subagents']['scoping']
print('real-time :', [p.split(':',1)[1] for p in sc['realtime']])
print('replay    :', [p.split(':',1)[1] for p in sc['historical']])
counts = {}
for v in sc['victims']:
    t,i = v.split(':',1); k = ID.classify_entity({'type':t,'id':i})
    counts[k] = counts.get(k,0)+1
print('identity kinds:', counts)

## 7. OSINT enrichment (Maltego · SpiderFoot · Criminal IP · SpyCloud · …)
Pluggable adapters — wire real API keys in `identities.py`.

In [ ]:
for kind, vals in (('ip', IOCS.ips), ('domain', IOCS.domains), ('hash', IOCS.hashes)):
    for val in list(vals)[:1]:
        print(kind, val, '->', list(ID.osint_enrich(val, kind).keys()))

## 8. Hypotheticals / what-if
Quantify the WildFire-loop's value, and test an arbitrary 'if this were C2' hypothesis.

In [ ]:
from pipeline import detect_verdict_fusion, detect_ioc_blast, IOCSet
vf = detect_verdict_fusion(norm)
known_wo = {e for f in vf for e in f.entities if e.split(':',1)[0] in ('host','identity')}
with_loop = g.reachable_principals(IOCS.keys())
print('victims WITHOUT loop:', len(known_wo), '| WITH loop:', len(with_loop))
ip = next(n['id'] for n in g.nodes.values() if n['type']=='ip' and n['id'].startswith('52.'))
hyp = detect_ioc_blast(norm, IOCSet(set(), {ip}, set(), set()))
print(f'HYPOTHESIS: if {ip} were C2 -> {len(hyp)} entities implicated')

## 9. Generate an incident report

In [ ]:
import report
norm2, g2, findings2, inv2, scores2, metrics2 = report.build()
md = report.markdown(findings2, inv2, scores2, metrics2)
print(md[:900])
open('investigation_report.md','w').write(md); print('\n...wrote investigation_report.md')

## 10. Write back to Abstract (field-set + view for this investigation)
Creates real tenant objects via the REST API (needs a write-scoped key).

In [ ]:
if client:
    fs = client.create_fieldset({'name':'[ABS-DEMO] Notebook investigation',
        'fields':['type','@timestamp','severity','user_name','source_address',
                  'file.hash.sha256','threat.technique_id','message'], 'tags':['abs-demo','notebook']})
    print('field-set:', (fs.get('body') or {}).get('id'), 'status', fs.get('status'))
else:
    print('offline — set ~/.abstract.env to enable write-back')

## The closed loop
```
 Abstract pipeline ──(API/MCP)──► this notebook ──(field-sets · views · suppressions · reports)──► Abstract
 triggers: new finding · new AIG IOC · hourly re-score · score-threshold → SOAR
```
Live: field-sets, views, suppressions, MITRE coverage. Simulated in-engine: replay, scoring,
prediction, sub-agents, incident narrative — see `DEMO-CATALOG.md` for the live-vs-simulated line.